# Excepciones, Persistencia y Testing Básico

## Tipos de Errores y Jerarquía (25 Minutos)

### 1.1 Errores de Sintaxis (`SyntaxError`) vs. Excepciones en Tiempo de Ejecución
* **Errores de Sintaxis (Parsing Errors):** Ocurren **antes** de que el programa empiece a ejecutarse. El analizador de Python detecta que el código viola la gramática del lenguaje.  
* **Excepciones (Runtime Exceptions):** El código está bien escrito gramaticalmente, pero ocurre un problema inesperado durante la ejecución (ej. dividir por cero, índice fuera de rango, archivo no encontrado).

---

### 1.2 La Jerarquía Oficial de Excepciones (Python Built-in Exceptions)
En Python, **todas las excepciones son clases** organizadas en un árbol jerárquico estricto. La jerarquía oficial de la documentación de Python ([docs.python.org/3/builtins/exceptions.html](https://docs.python.org/3/builtins/exceptions.html)) es la siguiente:

```text
BaseException
 ├── BaseExceptionGroup                 (Python 3.11+: agrupa múltiples excepciones)
 ├── GeneratorExit                      (Lanzada al cerrar un generador con .close())
 ├── KeyboardInterrupt                  (Ctrl + C del usuario en terminal)
 ├── SystemExit                         (sys.exit() para terminar el programa)
 └── Exception                          (Base de TODOS los errores normales de aplicación)
      ├── ArithmeticError               (Base de errores matemáticos)
      │    ├── FloatingPointError       (Fallo en cálculo de coma flotante de bajo nivel)
      │    ├── OverflowError            (Número excede el límite numérico representable)
      │    └── ZeroDivisionError        (División o módulo por cero)
      ├── AssertionError                (Fallo en sentencia 'assert')
      ├── AttributeError                (Acceso a un atributo o método inexistente en un objeto)
      ├── BufferError                   (Operación inválida sobre un búfer de memoria interna)
      ├── EOFError                      (input() llega al fin de archivo sin recibir datos)
      ├── ExceptionGroup                (Subclase de BaseExceptionGroup para Exception)
      ├── ImportError                   (Fallo al importar un módulo o símbolo)
      │    └── ModuleNotFoundError      (El módulo solicitado no existe en el entorno)
      ├── LookupError                   (Base de errores de búsqueda e indexación)
      │    ├── IndexError               (Índice fuera del rango en una lista/secuencia)
      │    └── KeyError                 (Clave no encontrada en un diccionario o mapeo)
      ├── MemoryError                   (El intérprete agotó la memoria RAM disponible)
      ├── NameError                     (Uso de variable, función o nombre no definido)
      │    └── UnboundLocalError        (Variable local referenciada antes de su asignación)
      ├── OSError                       (Errores del sistema operativo, archivos y red)
      │    ├── BlockingIOError          (Operación de E/S bloqueante no permitida)
      │    ├── ChildProcessError        (Fallo en subproceso del sistema operativo)
      │    ├── ConnectionError          (Base de problemas de conexión de red)
      │    │    ├── BrokenPipeError
      │    │    ├── ConnectionAbortedError
      │    │    ├── ConnectionRefusedError
      │    │    └── ConnectionResetError
      │    ├── FileExistsError          (Intento de crear un archivo/carpeta que ya existe)
      │    ├── FileNotFoundError        (Archivo o directorio no encontrado en la ruta)
      │    ├── InterruptedError         (Llamada al sistema interrumpida por una señal)
      │    ├── IsADirectoryError        (Se esperaba un archivo, pero es un directorio)
      │    ├── NotADirectoryError       (Se esperaba un directorio, pero es un archivo)
      │    ├── PermissionError          (Sin permisos suficientes de lectura o escritura)
      │    ├── ProcessLookupError       (Proceso inexistente en el sistema operativo)
      │    └── TimeoutError             (Tiempo límite de espera de la operación agotado)
      ├── ReferenceError                (Acceso a objeto de referencia débil ya destruido por el GC)
      ├── RuntimeError                  (Error genérico en tiempo de ejecución)
      │    ├── NotImplementedError      (Método abstracto no implementado aún)
      │    └── RecursionError           (Límite de recursión máxima alcanzado)
      ├── StopAsyncIteration            (Fin de iterador asíncrono)
      ├── StopIteration                 (Fin de iterador síncrono al llamar next())
      ├── SyntaxError                   (Error en la sintaxis del lenguaje)
      │    └── IndentationError         (Sangría o indentación incorrecta)
      │         └── TabError            (Mezcla inconsistente de tabs y espacios)
      ├── SystemError                   (Error interno del motor CPython)
      ├── TypeError                     (Operación entre tipos incompatibles)
      ├── ValueError                    (Tipo correcto, pero valor inválido o fuera de rango)
      │    └── UnicodeError             (Fallo de codificación/decodificación de texto)
      │         ├── UnicodeDecodeError
      │         ├── UnicodeEncodeError
      │         └── UnicodeTranslateError
      └── Warning                       (Base de avisos no fatales)
           ├── DeprecationWarning, FutureWarning, UserWarning, RuntimeWarning, etc.
```

---

### 1.3 `BaseException`

`BaseException` es la **raíz absoluta** de todo el árbol de excepciones de Python.

#### Separacion de `BaseException` de `Exception`
Python divide sus excepciones en dos grandes categorías conceptuales:
1. **Excepciones de Control del Sistema (`KeyboardInterrupt`, `SystemExit`, `GeneratorExit`):** No representan errores de programación ni de datos; son **instrucciones de bajo nivel** dirigidas al intérprete para detenerse o cerrar recursos.
   * `SystemExit`: Se lanza cuando un programa llama a `sys.exit()` para terminar ordenadamente.
   * `KeyboardInterrupt`: Se lanza cuando el usuario pulsa `Ctrl + C` en la consola para interrumpir un script colgado o un bucle.
   * `GeneratorExit`: Se lanza cuando un generador o corrutina se cierra invocando `.close()`.
2. **Excepciones de Aplicación (`Exception` y sus descendientes):** Representan anomalías o errores normales de ejecución en nuestra aplicación (`ValueError`, `TypeError`, `KeyError`, etc.).

#### `except:` vacío y `except BaseException:`
Si escribimos:
```python
try:
    ejecutar_servidor_o_tarea()
except Exception:
    print("Ocurrió un fallo en la lógica o datos de la aplicación.")

try:
    bucle_infinito()
except BaseException:
    print("¡No me puedo detener ni con Ctrl+C!")
```
* **Con `except Exception:`**: Si el usuario pulsa `Ctrl + C`, el script **se detiene correctamente** porque `KeyboardInterrupt` **no** hereda de `Exception`.
* **Con `except BaseException:` (o `except:` a secas)**: El programa interceptará el `Ctrl + C` del usuario y el `sys.exit()`, volviéndose inmune a la detención desde consola.


In [ ]:
class ValidacionDatoError(Exception):
    """Error estándar de la aplicación: se captura con 'except Exception'."""
    pass

class CancelacionCriticaWorker(BaseException):
    """Señal de control de bajo nivel: IGNORA 'except Exception' y se propaga."""
    pass

def procesar_tarea_worker(comando: str):
    print(f"\n--- Procesando comando: '{comando}' ---")
    if comando == "dato_invalido":
        raise ValidacionDatoError("El correo no tiene formato válido.")
    elif comando == "abortar_worker":
        raise CancelacionCriticaWorker("Señal del orquestador: abortar worker inmediatamente.")

for comando in ["dato_invalido", "abortar_worker"]:
    try:
        procesar_tarea_worker(comando)
    except Exception as err:
        print(f" s : {type(err).__name__} -> {err}")
    except BaseException as err_base:
        print(f" s : {type(err_base).__name__} -> {err_base}")


In [ ]:
import math

ejemplos_errores = [
    
    ("ZeroDivisionError", lambda: 10 / 0),
    ("OverflowError", lambda: math.exp(1000)),  
    
    ("ValueError", lambda: int("veinte")),
    ("TypeError", lambda: "Edad: " + 25),  
        
    ("KeyError", lambda: {"nombre": "Ana"}["edad"]),
    ("IndexError", lambda: [1, 2, 3][10]),
        
    ("AttributeError", lambda: "texto".append("!")),  
    ("NameError", lambda: variable_inexistente_xyz + 1),  
        
    ("ModuleNotFoundError", lambda: __import__("modulo_inexistente_12345")),
    ("FileNotFoundError", lambda: open("archivo_inexistente_xyz.txt", "r")),
        
    ("StopIteration", lambda: next(iter([]))),  
    ("AssertionError", lambda: (_ for _ in ()).throw(AssertionError("Aserción fallida"))),
]

print("=" * 75)
print(f"{'Excepción':<22} | {'Clase Padre Inmediata':<25} | {'Mensaje Original'}")
print("=" * 75)

for nombre, operacion in ejemplos_errores:
    try:
        operacion()
    except Exception as error:
        clase = type(error)
        padre = clase.__base__.__name__ if clase.__base__ else "None"
        msj = str(error)
        if len(msj) > 35:
            msj = msj[:32] + "..."
        print(f"{clase.__name__:<22} | (Hija de {padre:<17}) | {msj}")


---
## `try`, `except`, `else`, `finally` y `raise`

\ `try` y `except`, pero solo se requieren **4 clausuras** para mantener el bloque `try` lo más pequeño posible:

1. **`try`**: Contiene **únicamente** la línea o líneas que pueden fallar (principio de superficie mínima).
2. **`except`**: Se ejecuta **solo si** ocurre una excepción compatible dentro del `try`.
3. **`else`**: Se ejecuta **solo si NO ocurrió ninguna excepción** en el `try`. Evita capturar por accidente errores de código posterior.
4. **`finally`**: Se ejecuta **siempre** (haya error, no haya error, o incluso si hay un `return`), ideal para liberar recursos o registrar auditorías.

In [ ]:
def calcular_cuota_mensual(monto_str: str, meses: int) -> float | None:
    print(f"\n--- Procesando solicitud: monto='{monto_str}', meses={meses} ---")
    try:
        monto = float(monto_str)
        cuota = monto / meses
    except ValueError as err:
        print(f"Error de formato: '{monto_str}' no es válido ({err}).")
        return None
    except ZeroDivisionError as err:
        print(f"Error matemático: El número de meses no puede ser 0. {err}")
        return None
    else:
        print(f"Cálculo exitoso. Cuota base: ${cuota:,.2f}")
        return round(cuota, 2)
    finally:
        print("Auditoría de operación cerrada.")

calcular_cuota_mensual("1200.50", 12)
calcular_cuota_mensual("mil", 12)
calcular_cuota_mensual("1200", 0)


--- Procesando solicitud: monto='1200.50', meses=12 ---
Cálculo exitoso. Cuota base: $100.04
Auditoría de operación cerrada.

--- Procesando solicitud: monto='mil', meses=12 ---
Error de formato: 'mil' no es válido (could not convert string to float: 'mil').
Auditoría de operación cerrada.

--- Procesando solicitud: monto='1200', meses=0 ---
Error matemático: El número de meses no puede ser 0. <built-in method with_traceback of ZeroDivisionError object at 0x0000023752A470A0>
Auditoría de operación cerrada.


### 2.2 Buenas Practicas

#### A. Excepciones con `.add_note()`
Antes de Python 3.11, si querías agregar contexto a un error (como qué fila o usuario falló) tenías que crear una nueva excepción y perder o ensuciar el mensaje original. Ahora todas las excepciones tienen el método `.add_note("texto")`.

#### B. Captura Múltiple y Nota sobre Python 3.14
* **En Python 3.13 y anteriores:** `except (ValueError, TypeError) as e:`
* **En Python 3.14:** Cuando **no** usas `as e`, puedes omitir los paréntesis: `except ValueError, TypeError:`. *(Por compatibilidad hacia atrás con 3.13, los paréntesis siguen siendo válidos y recomendados al usar `as`)_.

#### C. Supresión Limpia con `contextlib.suppress`
En lugar de escribir un bloque `try...except FileNotFoundError: pass`, la comunidad moderna prefiere usar `with suppress(...)` de la librería estándar.

### With Statements

La sentencia `with` es una estructura de control diseñada para simplificar la gestión de recursos (archivos, conexiones de red, bloqueos de concurrencia, bases de datos, etc.). 
Su objetivo principal es garantizar la adquisición y liberación segura de recursos mediante el protocolo de **Gestores de Contexto** (*Context Managers*), evitando fugas de memoria o descriptores abiertos incluso si ocurren errores inesperados.

#### 1. El Problema que Resuelve
Antes de `with`, para abrir y cerrar un recurso de forma segura se requería un bloque `try ... finally`:

```python
# Enfoque antiguo y propenso a errores u omisiones
archivo = open("datos.txt", "w", encoding="utf-8")
try:
    archivo.write("Hola, mundo")
finally
    archivo.close()  # Obligatorio para asegurar el cierre

```
Si el desarrollador olvidaba el bloque `finally`, cualquier excepción intermedia dejaba el archivo abierto en el sistema operativo.
Con la sentencia `with`, el código se reduce a:

```python

with open("datos.txt", "w", encoding="utf-8") as archivo:
    archivo.write("Hola, mundo")

```

## 2. Context Protocol

Cualquier objeto utilizado en una cláusula `with` debe implementar dos métodos  del modelo de datos de Python:

1. `__enter__()`: Se ejecuta antes de entrar al bloque. Lo que retorne este método es lo que se asigna a la variable tras el `as`.
2. `__exit__(exc_type, exc_val, exc_tb)`: Se ejecuta **siempre** al salir del bloque, ya sea por terminación natural o por una excepción.


#### Ejemplo de implementación personalizada:

```python
# Manera de crear 
class TemporizadorSimple:
    
    def __enter__(self):        
        import time
        self.inicio = time.perf_counter()
        print("-> Cronómetro iniciado")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        import time
        self.fin = time.perf_counter()
        duracion = self.fin - self.inicio
        print(f"-> Cronómetro detenido. Duración: {duracion:.4f}s")
        return False

with TemporizadorSimple():
    suma = sum(range(1_000_000))

```


#### 3. Buenas Practicas

A partir de Python 3.10+, se permite envolver múltiples gestores de contexto en paréntesis para organizar mejor líneas largas:

```python

from pathlib import Path

origen = Path("origen.txt")
destino = Path("destino.txt")

with (
    origen.open("r", encoding="utf-8") as f_in,
    destino.open("w", encoding="utf-8") as f_out,
):
    f_out.write(f_in.read())

```



#### B. Creación rápida con `@contextlib.contextmanager`

Para no escribir clases completas con `__enter__` y `__exit__`, se puede usar un generador decorado:

```python

from contextlib import contextmanager

@contextmanager
def transaccion_simulada(nombre: str):
    print(f"[INICIO] Transacción: {nombre}")
    try:
        yield 
        print(f"[COMMIT] Transacción: {nombre} completada con éxito")

    except Exception as e:
        print(f"[ROLLBACK] Error detectado: {e}")
        raise
    finally:
        print(f"[FIN] Limpieza de recursos para {nombre}")

with transaccion_simulada("Transferencia #101"):
    print("Procesando datos...")

```

In [ ]:
from contextlib import suppress

class SaldoInsuficienteError(ValueError):    
    pass

def procesar_pago(usuario_id: str, saldo: float, monto: float) -> float:
    if monto <= 0:
        raise ValueError(f"El monto a pagar debe ser positivo, se recibió: {monto}")
    if monto > saldo:
        raise SaldoInsuficienteError(f"Faltan ${monto - saldo:.2f} para cubrir el pago.")
    return saldo - monto

lote_pagos = [
    {"id": "USR-001", "saldo": 500.0, "monto": 1500.0},
    {"id": "USR-002", "saldo": 100.0, "monto": 250.0},    
]

for indice, registro in enumerate(lote_pagos, start=1):
    try:
        nuevo_saldo = procesar_pago(registro["id"], registro["saldo"], registro["monto"])
        print(f"Pago procesado para {registro['id']}. Nuevo saldo: ${nuevo_saldo:.2f}")
    except Exception as err:        
        err.add_note(f"Contexto: Fallo ocurrido en el lote #{indice} para el usuario {registro['id']}")
        print(f"Excepción capturada: {type(err).__name__}: {err}")
        print(f"Notas adjuntas a la excepción: {err.__notes__}")

diccionario_temporal = {"clave_a": 100}
with suppress(KeyError):    
    del diccionario_temporal["clave_inexistente"]
print("\n Supresión limpia con contextlib.suppress ejecutada correctamente.")

Excepción capturada: SaldoInsuficienteError: Faltan $1000.00 para cubrir el pago.
Notas adjuntas a la excepción: ['Contexto: Fallo ocurrido en el lote #1 para el usuario USR-001']
Excepción capturada: SaldoInsuficienteError: Faltan $150.00 para cubrir el pago.
Notas adjuntas a la excepción: ['Contexto: Fallo ocurrido en el lote #2 para el usuario USR-002']

 Supresión limpia con contextlib.suppress ejecutada correctamente.


### 2.3 Grupos de Excepciones (`ExceptionGroup` y `except*`)
Cuando validamos un formulario o procesamos un lote de datos, es frustrante para el usuario que el programa falle en el primer error y oculte los demás. Con `ExceptionGroup` y `except*` podemos reportar **todos los errores a la vez**.

In [ ]:
def validar_registro_usuario(nombre: str, edad: int, correo: str) -> None:
    errores: list[Exception] = []
    
    if not nombre.strip():
        errores.append(ValueError("El nombre no puede estar vacío."))
    if not isinstance(edad, int):
        errores.append(TypeError("La edad debe ser un número entero."))
    elif edad < 18:
        errores.append(ValueError("El usuario debe ser mayor de edad (>= 18)."))
    if "@" not in correo:
        errores.append(ValueError(f"El correo '{correo}' no contiene '@'."))
        
    if errores:
        raise ExceptionGroup("Fallo en la validación del usuario", errores)

try:
    validar_registro_usuario(nombre="", edad="veinte", correo="correo_invalido.com")
except* ValueError as grupo_valor:
    for sub_error in grupo_valor.exceptions:
        print(f"   - {sub_error}")
except* TypeError as grupo_tipo:    
    for sub_error in grupo_tipo.exceptions:
        print(f"   - {sub_error}")

### Persistencia y Manejo de Archivos en Python

### 1. Fundamentos y Reglas de Oro
#### 1.1. Manejo de Rutas con `pathlib.Path`
Evita concatenar cadenas manuales (como `"carpeta/" + "archivo.txt"`), ya que los separadores de ruta varían entre sistemas operativos (`/` en POSIX frente a `\` en Windows).

* **Operador `/`:** Permite componer rutas de forma intuitiva: `ruta_archivo = RUTA_BASE / "subcarpeta" / "datos.json"`.
* **Métodos clave de `Path`:**
  * `.exists()`: Verifica si el archivo o carpeta existe.
  * `.is_file()` / `.is_dir()`: Comprueba el tipo de nodo.
  * `.mkdir(parents=True, exist_ok=True)`: Crea directorios intermedios sin lanzar error si ya existen.
  * `.resolve()`: Obtiene la ruta absoluta resolviendo enlaces simbólicos.
  * `.read_text(encoding="utf-8")` / `.write_text(texto, encoding="utf-8")`: Atajos para archivos pequeños donde no se requiere control granular del descriptor.


### 2. Modos de Apertura en `open()`

La función integrada `open(file, mode="r", buffering=-1, encoding=None, errors=None, newline=None, ...)` utiliza un sistema de banderas combinables para definir los permisos y el comportamiento del cursor.

#### 2.1. Modos Básicos de Operación

| Modo | Acción principal | ¿Crea archivo si no existe? | ¿Trunca (borra) el contenido previo? | Posición inicial del cursor |
| :--- | :--- | :---: | :---: | :--- |
| `'r'` | Lectura exclusiva | **No** (Lanza `FileNotFoundError`) | No | Inicio (byte 0) |
| `'w'` | Escritura exclusiva | **Sí** | **Sí** (vacía el archivo de inmediato) | Inicio (byte 0) |
| `'a'` | Añadir (*Append*) | **Sí** | No | Final del archivo |
| `'x'` | Creación exclusiva | **Sí** (Falla con `FileExistsError` si ya existe) | No | Inicio (byte 0) |



#### 2.2. Modificadores de Formato
* **`'t'` (Texto - por defecto):** Lee y escribe cadenas de texto (`str`). Realiza traducción automática de saltos de línea y requiere decodificación/codificación de bytes.
* **`'b'` (Binario):** Lee y escribe secuencias de bytes crudas (`bytes`). Se utiliza para imágenes, ejecutables, audio o formatos serializados como Pickle. **No admite el parámetro `encoding`**.

#### 2.3. Modificador de Actualización (`'+'`)
El símbolo `+` añade la capacidad contraria (lectura a modos de escritura, o escritura a modos de lectura):

* **`'r+'` (Lectura y Escritura):**
  * Requiere que el archivo exista previamente.
  * No trunca el contenido inicial.
  * El cursor inicia en el byte 0; escribir sobrescribe los bytes existentes sin desplazar el resto.

* **`'w+'` (Escritura y Lectura):**
  * Crea el archivo si no existe.
  * **Trunca el archivo a 0 bytes** al abrirlo; se pierde cualquier contenido previo.

* **`'a+'` (Añadir y Lectura):**
  * Crea el archivo si no existe.
  * Las lecturas respetan la posición del cursor, pero **toda operación de escritura salta automáticamente al final del archivo**, sin importar llamadas previas a `.seek()`.


### 3. Codificación de Caracteres (*Encodings*)
#### 3.1. ¿Qué es un *Encoding* en realidad?
Un procesador y un disco solo almacenan bits (`0` y `1`, agrupados en bytes de 8 bits: `00000000` a `11111111`, es decir, enteros de 0 a 255).

* **Unicode:** Es un estándar internacional que asigna a cada carácter abstracto del mundo un identificador numérico único llamado **Code Point** (por ejemplo, la letra `ñ` es `U+00F1`). Unicode **no** es un formato de archivo; es un mapa conceptual.
* **Encoding (Codificación):** Es el algoritmo matemático que traduce ese *Code Point* abstracto a una secuencia concreta de bytes para escribirlo en disco, o viceversa para leerlo.
* Depender del valor predeterminado del sistema operativo es peligroso:
  * Windows tradicionalmente usaba codificaciones regionales como `cp1252` o `iso-8859-1`.
  * Linux y macOS usan universalmente `utf-8`.
  * Si guardas un archivo con caracteres especiales en Linux (`utf-8`) y lo abres en Windows sin especificar codificación (`cp1252`), los bytes que representan la `ñ` (`0xC3 0xB1`) se interpretarán erróneamente como dos caracteres distintos (`Ã±`). Este fenómeno se conoce como **Mojibake**.

#### 3.2. Catálogo de Encodings Comunes
1. **`utf-8` (Estándar de la industria):**
   * Longitud variable (de 1 a 4 bytes por carácter).
   * Los primeros 128 caracteres (0 a 127) coinciden exactamente con ASCII en tamaño y valor (1 solo byte).
   * Caracteres latinos con tildes, eñes y símbolos europeos suelen ocupar 2 bytes.
   * Emojis y alfabetos asiáticos suelen ocupar 3 o 4 bytes.
   * Es el estándar obligatorio recomendado por PEP 597 y PEP 686.

2. **`ascii`:**
   * Utiliza únicamente 7 bits (valores de 0 a 127).
   * Solo admite el alfabeto inglés básico, números y caracteres de control (`\n`, `\t`).
   * Fallará de inmediato si encuentra cualquier tilde o carácter internacional.

3. **`latin-1` / `iso-8859-1`:**
   * Codificación fija de 1 byte (rango completo de 0 a 255).
   * Cubre la mayoría de los idiomas de Europa Occidental.
   * Curiosidad técnica: Nunca lanza `UnicodeDecodeError` durante la lectura porque los 256 posibles valores de un byte están asignados a un carácter. Sin embargo, decodificará erróneamente cualquier archivo generado en `utf-8`.

4. **`cp1252` (Windows ANSI):**
   * Superconjunto propietario de `latin-1` utilizado históricamente por defecto en sistemas Windows en español/inglés.
   * Asigna caracteres legibles (como el símbolo de euro `€` o comillas tipográficas) en el rango `0x80` a `0x9F`, donde `iso-8859-1` tenía caracteres de control invisibles.

5. **`utf-16` / `utf-32`:**
   * `utf-16` usa unidades de 2 o 4 bytes. Suele requerir una cabecera llamada BOM (*Byte Order Mark*, `\ufeff`) para indicar el orden de bytes (*Little Endian* vs *Big Endian*).
   * `utf-32` utiliza estrictamente 4 bytes para cada carácter, consumiendo significativamente más memoria y espacio en disco.



#### 3.3. Manejo de Errores de Codificación (`errors`)
Al abrir un archivo de texto, puedes controlar la respuesta ante bytes corruptos o inesperados:

* `errors="strict"` (por defecto): Lanza `UnicodeDecodeError` o `UnicodeEncodeError`.
* `errors="ignore"`: Descarta silenciosamente los caracteres conflictivos.
* `errors="replace"`: Sustituye el byte no reconocido por el carácter de reemplazo oficial (``, `\ufffd`).
* `errors="backslashreplace"`: Reemplaza el carácter no soportado por su secuencia de escape legible (`\xNN`, `\uNNNN`).

### 4. Métodos de Lectura, Escritura y Control de Cursor

#### 4.1. Lectura Eficiente de Archivos

* **`f.read(size=-1)`:** Lee todo el archivo en una sola cadena (o `size` caracteres/bytes). Peligroso en archivos masivos por consumo de RAM.
* **`f.readline()`:** Lee una única línea hasta encontrar `\n`.
* **`f.readlines()`:** Carga todas las líneas en una lista en memoria (`list[str]`).

* **Iteración directa sobre el objeto archivo (Recomendado):**
  ```python
  with open("servidor.log", "r", encoding="utf-8") as f:

      for linea in f:
          procesar(linea.rstrip("\n"))

  ```



#### 4.2. Control del Cursor: `tell()` y `seek()`

* **`f.tell()`:** Devuelve un entero con la posición actual del cursor (en bytes desde el inicio).
* **`f.seek(offset, whence=0)`:** Mueve el cursor a una posición específica.
  * `whence=0` (`os.SEEK_SET`): Relativo al inicio del archivo (offset >= 0).
  * `whence=1` (`os.SEEK_CUR`): Relativo a la posición actual del cursor.
  * `whence=2` (`os.SEEK_END`): Relativo al final del archivo (usualmente offset negativo para retroceder). 

### 5. Persistencia Estructurada (Librería Estándar)
#### 5.1. Formato JSON (`json`)

Ideal para estructuras anidadas (diccionarios, listas, números, booleanos y texto).

* **`dump()` vs `dumps()`:**
  * `json.dump(obj, f)`: Serializa y escribe directamente en un archivo abierto.
  * `json.dumps(obj)`: Serializa a una cadena `str` en memoria.

* **`load()` vs `loads()`:**
  * `json.load(f)`: Lee y deserializa directamente desde un archivo.
  * `json.loads(cadena_json)`: Deserializa desde un `str`.

* **Parámetros obligatorios para legibilidad e internacionalización:**
  * `ensure_ascii=False`: Evita que caracteres como la `ñ` o tildes se escapen a formato `\u00f1`.
  * `indent=4`: Formatea el archivo con sangría legible.

#### 5.2. Formato CSV (`csv`)
Diseñado para datos tabulares (filas y columnas).

* **Regla crítica en Windows:** Siempre se debe pasar `newline=""` a `open()`. El módulo `csv` maneja sus propios saltos de línea; si no se especifica, Windows duplicará los retornos de carro produciendo filas vacías (`\r\r\n`).
* **Lectura/Escritura con Diccionarios:**
  * `csv.DictReader(f)`: Mapea cada fila a un diccionario ordenado usando la cabecera como claves.
  * `csv.DictWriter(f, fieldnames=[...])`: Escribe diccionarios verificando las columnas. Requiere llamar a `.writeheader()`.


In [ ]:
import json
from pathlib import Path

class GestorInventarioArchivo:    
    def __init__(self, ruta_archivo: Path) -> None:
        self.ruta = ruta_archivo

    def guardar_inventario(self, datos: dict[str, int]) -> None:
        """Guarda el diccionario validando tipos y asegurando codificación UTF-8."""
        if not isinstance(datos, dict):
            raise TypeError("El inventario debe ser un diccionario.")
        
        for producto, cantidad in datos.items():
            if not isinstance(cantidad, int) or cantidad < 0:
                raise ValueError(f"Cantidad inválida ({cantidad}) para el producto '{producto}'.")

        try:            
            self.ruta.parent.mkdir(parents=True, exist_ok=True)
            with self.ruta.open(mode="w", encoding="utf-8") as archivo:
                json.dump(datos, archivo, ensure_ascii=False, indent=4)
            return None
        except PermissionError as err:
            err.add_note(f"No se tienen permisos de escritura en: {self.ruta}")
            raise
        except OSError as err:
            err.add_note(f"Error del sistema operativo al escribir en: {self.ruta}")
            raise

    def cargar_inventario(self) -> dict[str, int]:
        """Lee el archivo JSON de forma segura o retorna un error enriquecido."""
        try:
            with self.ruta.open(mode="r", encoding="utf-8") as archivo:
                contenido = json.load(archivo)
                if not isinstance(contenido, dict):
                    raise ValueError("El archivo JSON no contiene un objeto raíz válido.")
                return contenido
        except FileNotFoundError as err:
            err.add_note(f"El archivo de persistencia '{self.ruta.name}' aún no ha sido creado.")
            raise
        except json.JSONDecodeError as err:
            err.add_note(f"El archivo '{self.ruta.name}' está corrupto o mal formateado.")
            raise

ruta_demo = Path("datos_clase") / "inventario_tienda.json"
gestor = GestorInventarioArchivo(ruta_demo)

inventario_inicial = {"Café de Nariño": 25, "Piña Miel": 14, "Teclado Mecánico Español": 8}
gestor.guardar_inventario(inventario_inicial)

datos_recuperados = gestor.cargar_inventario()

In [ ]:
ruta_origen = Path("datos_clase") / "inventario_tienda.json"
ruta_respaldo = Path("datos_clase") / "inventario_respaldo.txt"

try:
    with (
        ruta_origen.open("r", encoding="utf-8") as lector,
        ruta_respaldo.open("w", encoding="utf-8") as escritor,
    ):
        datos = json.load(lector)
        escritor.write("=== REPORTE DE INVENTARIO ===\n")
        for producto, stock in datos.items():
            escritor.write(f"• {producto}: {stock} unidades\n")
except OSError as e:
    print(f"Error procesando archivos: {e}")
else:
    print(ruta_respaldo.read_text(encoding="utf-8"))

---
## Testing 

Probar codigo es esencial.

1. **La sentencia `assert`**: Ideal para verificaciones rápidas durante el aprendizaje.
2. **El módulo `unittest`**: El framework de pruebas oficial integrado en Python.
   * `self.assertEqual(a, b)`: Verifica que el resultado sea el esperado.
   * `with self.assertRaises(TipoExcepcion):`: **¡Fundamental!** Verifica que nuestro código lance la excepción correcta cuando recibe datos inválidos.
3. **Aislamiento con `tempfile.TemporaryDirectory`**: Una práctica profesional clave. Las pruebas automatizadas **nunca deben ensuciar el disco duro real**; crean una carpeta temporal que se autodestruye al terminar el test.

In [ ]:
import tempfile
import unittest

class TestGestorInventario(unittest.TestCase):
    """
    Suite de pruebas unitarias para verificar tanto los casos de éxito
    como el lanzamiento correcto de excepciones y la persistencia en archivos.
    """

    def setUp(self) -> None:
        """Se ejecuta antes de CADA test. Crea un directorio temporal aislado."""
        self.dir_temporal = tempfile.TemporaryDirectory()
        self.ruta_prueba = Path(self.dir_temporal.name) / "test_inventario.json"
        self.gestor = GestorInventarioArchivo(self.ruta_prueba)

    def tearDown(self) -> None:
        """Se ejecuta después de CADA test (como un finally). Limpia archivos."""
        self.dir_temporal.cleanup()

    def test_guardar_y_cargar_exitoso(self) -> None:
        """Prueba el camino feliz (Happy Path) de persistencia UTF-8."""
        datos_esperados = {"Monitor 27\"": 5, "Ratón Inalámbrico": 12}
        self.gestor.guardar_inventario(datos_esperados)
        
        # Verificamos que el archivo físico exista
        self.assertTrue(self.ruta_prueba.exists()) 
        
        # Verificamos que los datos leídos sean idénticos a los guardados
        datos_obtenidos = self.gestor.cargar_inventario()
        self.assertEqual(datos_esperados, datos_obtenidos)

    def test_error_al_cargar_archivo_inexistente(self) -> None:
        """Verifica que se lance FileNotFoundError si el archivo no existe."""
        with self.assertRaises(FileNotFoundError) as contexto:
            self.gestor.cargar_inventario()
        
        # Verificamos incluso que nuestra nota moderna (.add_note) esté presente
        notas = getattr(contexto.exception, "__notes__", [])
        self.assertTrue(any("aún no ha sido creado" in nota for nota in notas))

    def test_error_cantidad_negativa(self) -> None:
        """Verifica que no se permita guardar stock negativo (ValueError)."""
        inventario_invalido = {"Laptop": -3}
        with self.assertRaises(ValueError):
            self.gestor.guardar_inventario(inventario_invalido)

    def test_error_archivo_json_corrupto(self) -> None:
        """Simula un archivo dañado y verifica que lance JSONDecodeError."""
        self.ruta_prueba.write_text("{json_invalido: sin_comillas", encoding="utf-8")
        with self.assertRaises(json.JSONDecodeError):
            self.gestor.cargar_inventario()

# Ejecución de unittest adaptada para Jupyter Notebooks (argv=[''], exit=False)
suite = unittest.TestLoader().loadTestsFromTestCase(TestGestorInventario)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)